# Data-derived covariates (p2) -- control genes, RUVg factors, RDA, VIF

1. Control genes: detected in every batch and least associated with phenotype (all samples, disease included)
2. Overlap of the control set with PanglaoDB hematopoietic cell types (Platelets = the Platelet Score genes)
3. RUVg factors W from HC only (`RUVSeq::RUVg` algorithm, checked against R), projected onto every other sample
4. RDA: explained variance of HC expression by BIAS10, W_k and both, and the batch variance left over
5. VIF of W_1..k together with BIAS10

Everything here is fitted on all engine HC at once and is for choosing (control-gene rule, N, k) only.
The p2 provider refits W inside every CV fold / LOBO training set.
The h5ad `RUVg_Platelet_*` / `Proposed_Full_*` layers are per-study fits and are never used.

Outputs: `results/covariates/` (tables, W) and `results/Figures/covariate_*.png`.

In [80]:
import subprocess
import sys
import tempfile
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import f as f_dist
from scipy.stats import false_discovery_control, fisher_exact

parent_dir = str(Path.cwd().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from viz_style import FS_ANNOT_SM, apply_style
apply_style()

import _FinalModelingMethodImprovement.config as config
from _FinalModelingMethodImprovement.engine.validation.lobo_engine import _decode_cat, load_full_data

OUT = config.COVARIATE_DIR
FIG_DIR = config.RESULTS_DIR / 'Figures'
OUT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DETECT_MIN = 0.95
MIN_BATCH_N = 10
N_CONTROLS = [100, 200, 300, 500, 1000]
N_MAIN = 1000
K_MAX = 10
pd.set_option('display.width', 250)

In [81]:
data = load_full_data()
genes = list(data['gene_col'])
Y = data['Y'][:, [data['gene_col'][g] for g in genes]]
lib = data['Y'].sum(1)
del data['Y']
logcpm = np.log2(Y / lib[:, None] * 1e6 + 1)
with h5py.File(config.H5AD_PATH, 'r') as f:
    sym_of = dict(zip(f['var/_index'][()].astype(str), _decode_cat(f['var/GeneName'])))
modeled = set(pd.read_csv(config.ENGINE_MIXED_DIR / 'training_summary.csv', usecols=['gene'])['gene'])
batch, pheno = data['batch'], data['phenotype']
hc_fit = data['is_hc'] & ~np.isin(batch, list(data['small_hc_batches']))
print(f'samples {len(batch)}, HC used for W {hc_fit.sum()}, protein-coding genes {len(genes)}, modeled {len(modeled)}')

samples 1606, HC used for W 679, protein-coding genes 20097, modeled 20097


## 1. Control genes

Detection: count > 0 in at least `DETECT_MIN` of the samples of every batch with >= `MIN_BATCH_N` samples.
Phenotype association: per gene, F-test of `logCPM ~ batch + phenotype` against `logCPM ~ batch` over all samples,
so only within-batch phenotype contrasts count. Control set = the N detected, modeled genes with the largest p (equivalently the smallest phenotype partial R^2,
`pheno_r2` = share of the within-batch variance explained by phenotype, since every gene has the same df).

Limitation (accepted): a large group-level p is weak evidence of no disease association -- most batches hold a
single phenotype, the within-batch contrasts come from few batches, and non-significance is not absence of effect.

In [82]:
def basis(X):
    u, s, _ = np.linalg.svd(X, full_matrices=False)
    return u[:, s > s[0] * 1e-10]


def rss(Q, Yc):
    return (Yc ** 2).sum(0) - ((Q.T @ Yc) ** 2).sum(0)


STATS_CACHE = OUT / 'control_gene_stats.csv'
if STATS_CACHE.is_file():
    stats = pd.read_csv(STATS_CACHE, index_col='gene')
else:
    big = pd.Series(batch).value_counts().loc[lambda v: v >= MIN_BATCH_N].index
    det = np.min([(Y[batch == b] > 0).mean(0) for b in big], axis=0)
    B = pd.get_dummies(batch).values.astype(float)
    P = pd.get_dummies(pheno).values.astype(float)
    Q0, Q1 = basis(B), basis(np.hstack([B, P]))
    df1, df2 = Q1.shape[1] - Q0.shape[1], len(batch) - Q1.shape[1]
    r0, r1 = rss(Q0, logcpm), rss(Q1, logcpm)
    with np.errstate(invalid="ignore", divide="ignore"):
        F = ((r0 - r1) / df1) / (r1 / df2)
        r2p = (r0 - r1) / r0
    stats = pd.DataFrame(dict(symbol=[sym_of.get(g, '') for g in genes], min_batch_detect=det,
                              mean_logcpm=logcpm.mean(0), F=F, p=f_dist.sf(F, df1, df2), pheno_r2=r2p,
                              modeled=[g in modeled for g in genes]), index=pd.Index(genes, name='gene'))
    stats.to_csv(STATS_CACHE)

mix = pd.crosstab(batch, pheno == 'Healthy Control').loc[lambda t: (t > 0).all(1)]
print(f'batches with both HC and disease: {len(mix)} ({mix.values.sum()} samples)')
cand = stats[(stats['min_batch_detect'] >= DETECT_MIN) & stats['modeled']].sort_values('p', ascending=False)
controls = {n: cand.index[:n].tolist() for n in N_CONTROLS}
for n, c in controls.items():
    (OUT / f'control_genes_N{n}.txt').write_text('\n'.join(c))
print(f'detected + modeled: {len(cand)} of {len(stats)}')
display(pd.DataFrame({t: dict(n_genes=int(((stats['min_batch_detect'] >= t) & stats['modeled']).sum()),
                              n_p_gt_0_05=int(((stats['min_batch_detect'] >= t) & stats['modeled'] & (stats['p'] > 0.05)).sum()))
                      for t in (0.5, 0.7, 0.8, 0.9, 0.95)}).T.rename_axis('DETECT_MIN'))
summ = {n: dict(min_p_in_set=stats.loc[c, 'p'].min(), median_pheno_r2=stats.loc[c, 'pheno_r2'].median(),
                max_pheno_r2=stats.loc[c, 'pheno_r2'].max(), median_mean_logcpm=stats.loc[c, 'mean_logcpm'].median())
        for n, c in controls.items()}
summ['all detected'] = dict(min_p_in_set=np.nan, median_pheno_r2=cand['pheno_r2'].median(), max_pheno_r2=cand['pheno_r2'].max(),
                            median_mean_logcpm=cand['mean_logcpm'].median())
display(pd.DataFrame(summ).T)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
bins = np.linspace(0, cand['pheno_r2'].max(), 60)
axes[0].hist(cand['pheno_r2'], bins=bins, color='0.75', label='detected')
for n, col in zip(sorted(N_CONTROLS, reverse=True), ['#E69F00', '#0072B2', '#000000']):
    axes[0].hist(stats.loc[controls[n], 'pheno_r2'], bins=bins, histtype='step', color=col, label=f'control N={n}')
axes[0].set_xlabel('Phenotype partial R2 (within batch)'); axes[0].set_ylabel('Genes')
axes[0].legend(frameon=False, fontsize=FS_ANNOT_SM)
axes[1].scatter(cand['mean_logcpm'], -np.log10(cand['p'].clip(lower=1e-300)), s=2, color='0.75', label='detected')
for n, col in zip(sorted(N_CONTROLS, reverse=True), ['#E69F00', '#0072B2', '#000000']):
    c = stats.loc[controls[n]]
    axes[1].scatter(c['mean_logcpm'], -np.log10(c['p']), s=2, color=col, label=f'control N={n}')
axes[1].set_xlabel('Mean log2 CPM'); axes[1].set_ylabel('-log10 p')
axes[1].legend(frameon=False, fontsize=FS_ANNOT_SM, markerscale=4)
fig.tight_layout()
fig.savefig(FIG_DIR / 'covariate_control_genes.png', dpi=300, bbox_inches='tight')
plt.show()

batches with both HC and disease: 26 (1399 samples)
detected + modeled: 1227 of 20097


,n_genes,n_p_gt_0_05
DETECT_MIN,,
0.50,7101,114
0.70,4661,52
0.80,3419,27
0.90,2018,7
0.95,1227,2


,min_p_in_set,median_pheno_r2,max_pheno_r2,median_mean_logcpm
100,2.740133e-10,0.044169,0.054187,6.718254
200,3.771972e-15,0.054244,0.070555,6.825995
300,5.720430e-20,0.062731,0.085832,6.856469
500,2.262732e-28,0.078741,0.111027,6.683278
1000,9.009550e-47,0.111035,0.162033,6.415928
all detected,NaN,0.123406,0.225017,6.584789


/tmp/ipykernel_388147/3782585724.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Overlap with PanglaoDB hematopoietic cell types

Cell types of organ Blood / Immune system plus Hematopoietic stem cells (human markers).
Universe = detected, modeled genes. One-sided Fisher test (enrichment), BH across cell types x N.
`n_markers` = PanglaoDB human markers matched to a protein-coding gene; `n_detected` = those passing the detection rule
(the Fisher universe). Heatmap: cell types with >= 5 detected markers; label n = all markers of the set,
cell text = control genes in the set, colour = share of the detected markers.

In [83]:
pg = pd.read_csv(config.PANGLAO_TSV, sep='\t')
pg = pg[pg['species'].str.contains('Hs') & (pg['organ'].isin(['Blood', 'Immune system']) | (pg['cell type'] == 'Hematopoietic stem cells'))]
univ = set(cand['symbol']) - {''}
pc_sym = set(stats['symbol']) - {''}
n_markers = {ct: len(set(g['official gene symbol']) & pc_sym) for ct, g in pg.groupby('cell type')}
sets = {ct: set(g['official gene symbol']) & univ for ct, g in pg.groupby('cell type')}
rows = []
for n, c in controls.items():
    cs = set(stats.loc[c, 'symbol']) - {''}
    for ct, s in sets.items():
        a = len(cs & s)
        tab = [[a, len(cs) - a], [len(s) - a, len(univ) - len(cs) - len(s) + a]]
        odds, p = fisher_exact(tab, alternative='greater')
        rows.append(dict(N=n, cell_type=ct, n_markers=n_markers[ct], n_detected=len(s), overlap=a,
                         frac_of_markers=a / max(n_markers[ct], 1), frac_of_detected=a / max(len(s), 1),
                         jaccard=a / len(cs | s), odds_ratio=odds, p=p))
ov = pd.DataFrame(rows)
ov['q'] = false_discovery_control(ov['p'])
ov.to_csv(OUT / 'control_panglao_overlap.csv', index=False)
display(ov[ov['N'] == N_MAIN].sort_values('p').round(4).head(15))

mat = ov.pivot(index='cell_type', columns='N', values='frac_of_detected')
q = ov.pivot(index='cell_type', columns='N', values='q').loc[mat.index]
cnt = ov.pivot(index='cell_type', columns='N', values='overlap').loc[mat.index]
size = ov.groupby('cell_type')[['n_detected', 'n_markers']].first()
order = mat[size['n_detected'].reindex(mat.index) >= 5][N_MAIN].sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(4.5, 0.28 * len(order) + 1))
im = ax.imshow(mat.loc[order].values, cmap='Greys', aspect='auto', vmin=0)
for i, ct in enumerate(order):
    for j, n in enumerate(mat.columns):
        v = mat.loc[ct, n]
        ax.text(j, i, f"{int(cnt.loc[ct, n])}{'*' if q.loc[ct, n] < 0.05 else ''}", ha='center', va='center',
                fontsize=FS_ANNOT_SM, color='white' if v > 0.5 else 'black')
ax.set_xticks(range(len(mat.columns))); ax.set_xticklabels([f'N={n}' for n in mat.columns])
ax.set_yticks(range(len(order))); ax.set_yticklabels([f"{ct} (n={size.loc[ct, 'n_markers']})" for ct in order], fontsize=FS_ANNOT_SM)
fig.colorbar(im, ax=ax, label='Fraction of detected markers in control set')
ax.set_title('Control set vs PanglaoDB\n(cell: overlapping genes, * BH q < 0.05)')
fig.savefig(FIG_DIR / 'covariate_panglao_overlap.png', dpi=300, bbox_inches='tight')
plt.show()

,N,cell_type,n_markers,n_detected,overlap,frac_of_markers,frac_of_detected,jaccard,odds_ratio,p,q
141,1000,Platelets,126,37,36,0.2857,0.9730,0.0360,8.4398,0.0044,0.1165
127,1000,Erythroid-like and erythroid precursor cells,70,21,21,0.3000,1.0000,0.0210,inf,0.0131,0.2661
133,1000,Monocytes,99,29,28,0.2828,0.9655,0.0280,6.5103,0.0191,0.3182
132,1000,Megakaryocytes,47,10,10,0.2128,1.0000,0.0100,inf,0.1282,1.0000
137,1000,Neutrophils,69,8,8,0.1159,1.0000,0.0080,inf,0.1936,1.0000
148,1000,T memory cells,52,13,12,0.2308,0.9231,0.0120,2.7449,0.2750,1.0000
131,1000,Mast cells,137,5,5,0.0365,1.0000,0.0050,inf,0.3589,1.0000
126,1000,Erythroblasts,26,5,5,0.1923,1.0000,0.0050,inf,0.3589,1.0000
128,1000,Gamma delta T cells,37,5,5,0.1351,1.0000,0.0050,inf,0.3589,1.0000
124,1000,Dendritic cells,118,21,18,0.1525,0.8571,0.0179,1.3686,0.4366,1.0000


/tmp/ipykernel_388147/1381755344.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. RUVg factors

`RUVSeq::RUVg(isLog=TRUE, center=TRUE)`: W = left singular vectors of the HC control matrix centred by the HC gene means.
Other samples: W_new = (x_ctrl - HC mean) V D^-1, which reproduces W exactly on the HC.
The R check compares Python and R W column by column (|r|, sign is arbitrary).

In [84]:
def ruvg_fit(x_ctrl_hc, k):
    m = x_ctrl_hc.mean(0)
    u, d, vt = np.linalg.svd(x_ctrl_hc - m, full_matrices=False)
    return dict(mean=m, v=vt[:k].T, d=d[:k], var_frac=d ** 2 / (d ** 2).sum())


def ruvg_transform(fit, x_ctrl):
    return (x_ctrl - fit['mean']) @ fit['v'] / fit['d']


col = {g: i for i, g in enumerate(genes)}
fits, W = {}, {}
for n, c in controls.items():
    idx = [col[g] for g in c]
    fits[n] = ruvg_fit(logcpm[hc_fit][:, idx], K_MAX)
    W[n] = ruvg_transform(fits[n], logcpm[:, idx])
    pd.DataFrame(W[n], index=data['names'], columns=[f'W_{i + 1}' for i in range(K_MAX)]).assign(
        hc_fit=hc_fit, batch=batch, phenotype=pheno).to_csv(OUT / f'W_N{n}.csv')

R_CHECK = OUT / f'ruvg_r_check_N{N_MAIN}_det{DETECT_MIN}.csv'
rc = pd.read_csv(R_CHECK) if R_CHECK.is_file() else None
if rc is None or len(rc) != K_MAX:
    with tempfile.TemporaryDirectory() as tmp:
        idx = [col[g] for g in controls[N_MAIN]]
        pd.DataFrame(logcpm[hc_fit][:, idx].T).to_csv(f'{tmp}/x.csv', index=False)
        r = ('suppressMessages(library(RUVSeq)); x <- as.matrix(read.csv(commandArgs(TRUE)[1])); '
             f'w <- RUVg(x, seq_len(nrow(x)), k = {K_MAX}, isLog = TRUE, center = TRUE)$W; '
             'write.csv(w, commandArgs(TRUE)[2], row.names = FALSE)')
        subprocess.run([str(config.RUVSEQ_RSCRIPT), '-e', r, f'{tmp}/x.csv', f'{tmp}/w.csv'], check=True)
        w_r = pd.read_csv(f'{tmp}/w.csv').values
    rc = pd.DataFrame(dict(W=[f'W_{i + 1}' for i in range(K_MAX)],
                           abs_r=[abs(np.corrcoef(W[N_MAIN][hc_fit][:, i], w_r[:, i])[0, 1]) for i in range(K_MAX)],
                           max_abs_diff=[np.abs(np.abs(W[N_MAIN][hc_fit][:, i]) - np.abs(w_r[:, i])).max() for i in range(K_MAX)]))
    rc.to_csv(R_CHECK, index=False)
display(rc.round(6))

fig, ax = plt.subplots(figsize=(4.5, 3.5))
for n, fit in fits.items():
    ax.plot(range(1, K_MAX + 1), 100 * fit['var_frac'][:K_MAX], marker='o', ms=3, label=f'N={n}')
ax.set_xlabel('Component'); ax.set_ylabel('% of control-gene variance (HC)')
ax.legend(frameon=False, fontsize=FS_ANNOT_SM)
fig.savefig(FIG_DIR / 'covariate_ruvg_scree.png', dpi=300, bbox_inches='tight')
plt.show()

,W,abs_r,max_abs_diff
0,W_1,1.0,0.0
1,W_2,1.0,0.0
2,W_3,1.0,0.0
3,W_4,1.0,0.0
4,W_5,1.0,0.0
5,W_6,1.0,0.0
6,W_7,1.0,0.0
7,W_8,1.0,0.0
8,W_9,1.0,0.0
9,W_10,1.0,0.0


/tmp/ipykernel_388147/2185153382.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. RDA -- explained variance of HC expression

Y = HC log2 CPM of the modeled genes minus every control gene of the largest N (same Y for all N, no circularity).
Multivariate adjusted R^2 (`EDA/analysis/pipeline.py` definition). `W_k unique` = R^2(BIAS10 + W_k) - R^2(BIAS10);
`batch left` = R^2(covariates + Batch_ID) - R^2(covariates), the batch variance the covariates do not absorb.

Stacked bars, one per case (BIAS10 only, then BIAS10 + W_1..k): BIAS10 unique = R^2(BIAS10 + W) - R^2(W),
W unique = R^2(BIAS10 + W) - R^2(BIAS10), shared = the rest of R^2(BIAS10 + W); number on top = R^2(BIAS10 + W).

In [85]:
def adj_r2(Yc, X):
    Q = basis(np.column_stack([np.ones(len(Yc)), X]) if X is not None else np.ones((len(Yc), 1)))
    sst = (Yc ** 2).sum()
    r2 = 1 - rss(Q, Yc).sum() / sst
    n, p = len(Yc), Q.shape[1] - 1
    return max(0.0, 1 - (1 - r2) * (n - 1) / (n - p - 1))


RDA_CACHE = OUT / 'rda_hc.csv'
rda = pd.read_csv(RDA_CACHE) if RDA_CACHE.is_file() else None
if rda is None or set(rda['N']) != {0, *N_CONTROLS} or rda['k'].max() != K_MAX or rda['detect_min'].iloc[0] != DETECT_MIN:
    excl = set(controls[max(N_CONTROLS)])
    ycols = [col[g] for g in genes if g in modeled and g not in excl]
    Yc = logcpm[hc_fit][:, ycols]
    Yc = Yc - Yc.mean(0)
    Xb = data['X_raw'][hc_fit]
    Xb = (Xb - Xb.mean(0)) / Xb.std(0)
    Bt = pd.get_dummies(batch[hc_fit]).values.astype(float)
    base = dict(bias=adj_r2(Yc, Xb), batch=adj_r2(Yc, Bt))
    rows = [dict(N=0, k=0, W=0.0, bias_W=base['bias'], bias=base['bias'], batch=base['batch'],
                 batch_left=adj_r2(Yc, np.hstack([Xb, Bt])) - base['bias'])]
    for n in N_CONTROLS:
        Wh = W[n][hc_fit]
        for k in range(1, K_MAX + 1):
            bw = adj_r2(Yc, np.hstack([Xb, Wh[:, :k]]))
            rows.append(dict(N=n, k=k, W=adj_r2(Yc, Wh[:, :k]), bias_W=bw, bias=base['bias'], batch=base['batch'],
                             batch_left=adj_r2(Yc, np.hstack([Xb, Wh[:, :k], Bt])) - bw))
    rda = pd.DataFrame(rows)
    rda['W_unique'] = rda['bias_W'] - rda['bias']
    rda['bias_unique'] = rda['bias_W'] - rda['W']
    rda['detect_min'] = DETECT_MIN
    rda.to_csv(RDA_CACHE, index=False)
display(rda.round(4))

PARTS = [('BIAS10 unique', '#D55E00'), ('Shared', '#CC79A7'), ('W unique', '#0072B2'),
         ('Batch left over', '#E69F00'), ('Unexplained', '#E5E5E5')]
b0 = rda[rda['N'] == 0].assign(k=0)
fig, axes = plt.subplots(1, len(N_CONTROLS), figsize=(4.2 * len(N_CONTROLS), 4), sharey=True, squeeze=False)
for ax, n in zip(axes[0], N_CONTROLS):
    d = pd.concat([b0, rda[rda['N'] == n]]).set_index('k')
    parts = pd.DataFrame({'BIAS10 unique': d['bias_unique'], 'Shared': d['bias_W'] - d['bias_unique'] - d['W_unique'],
                          'W unique': d['W_unique'], 'Batch left over': d['batch_left']})
    parts['Unexplained'] = 1 - parts.sum(1)
    bottom = np.zeros(len(parts))
    for name, color in PARTS:
        ax.bar(range(len(parts)), 100 * parts[name], bottom=bottom, color=color, edgecolor='white', lw=0.4, label=name)
        bottom += 100 * parts[name].values
    for x, (v, top) in enumerate(zip(d['bias_W'], d['bias_W'] + d['batch_left'])):
        ax.text(x, 100 * top + 1, f'{100 * v:.0f}', ha='center', va='bottom', fontsize=FS_ANNOT_SM)
    ax.set_xticks(range(len(parts))); ax.set_xticklabels(parts.index)
    ax.set_xlabel('k (0 = BIAS10 only)'); ax.set_title(f'N = {n} control genes')
axes[0, 0].set_ylabel('HC expression variance (adjusted R2, %)')
axes[0, -1].legend(frameon=False, fontsize=FS_ANNOT_SM, loc='upper left', bbox_to_anchor=(1.02, 1))
fig.tight_layout()
fig.savefig(FIG_DIR / 'covariate_rda.png', dpi=300, bbox_inches='tight')
plt.show()

,N,k,W,bias_W,bias,batch,batch_left,W_unique,bias_unique,detect_min
0,0,0,0.0000,0.3559,0.3559,0.2273,0.0573,0.0000,0.3559,0.95
1,100,1,0.0582,0.3757,0.3559,0.2273,0.0527,0.0198,0.3175,0.95
2,100,2,0.2346,0.3835,0.3559,0.2273,0.0500,0.0276,0.1489,0.95
3,100,3,0.2649,0.4011,0.3559,0.2273,0.0390,0.0452,0.1362,0.95
4,100,4,0.2922,0.4061,0.3559,0.2273,0.0363,0.0501,0.1139,0.95
5,100,5,0.3038,0.4104,0.3559,0.2273,0.0348,0.0544,0.1065,0.95
6,100,6,0.3183,0.4169,0.3559,0.2273,0.0300,0.0610,0.0986,0.95
7,100,7,0.3332,0.4207,0.3559,0.2273,0.0288,0.0648,0.0875,0.95
8,100,8,0.3446,0.4234,0.3559,0.2273,0.0283,0.0675,0.0788,0.95
9,100,9,0.3576,0.4275,0.3559,0.2273,0.0280,0.0715,0.0698,0.95


/tmp/ipykernel_388147/345193740.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. VIF and correlation with BIAS10 (HC, N = N_MAIN)

W columns are orthogonal on the HC by construction, so VIF > 1 comes from overlap with BIAS10.

In [86]:
Xb = pd.DataFrame(data['X_raw'][hc_fit], columns=config.BIAS_COLUMNS)
Wm = pd.DataFrame(W[N_MAIN][hc_fit], columns=[f'W_{i + 1}' for i in range(K_MAX)])
vif = {}
for k in range(0, K_MAX + 1):
    c = pd.concat([Xb, Wm.iloc[:, :k]], axis=1).corr().values
    vif[k] = pd.Series(np.diag(np.linalg.inv(c)), index=list(Xb.columns) + list(Wm.columns[:k]))
vif = pd.DataFrame(vif).loc[vif[K_MAX].index]
vif.columns.name = 'k'
vif.to_csv(OUT / f'vif_N{N_MAIN}.csv')
display(vif.round(2))

corr = pd.concat([Xb, Wm], axis=1).corr().loc[Wm.columns, Xb.columns]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw=dict(width_ratios=[1, 1.2]))
im = axes[0].imshow(np.log10(vif.values.astype(float)), cmap='Reds', aspect='auto', vmin=0)
axes[0].set_xticks(range(vif.shape[1])); axes[0].set_xticklabels(vif.columns)
axes[0].set_yticks(range(vif.shape[0])); axes[0].set_yticklabels(vif.index, fontsize=FS_ANNOT_SM)
for i in range(vif.shape[0]):
    for j in range(vif.shape[1]):
        if np.isfinite(vif.values[i, j]):
            axes[0].text(j, i, f'{vif.values[i, j]:.1f}', ha='center', va='center', fontsize=6)
axes[0].set_xlabel('k (W included)'); axes[0].set_title('VIF (colour: log10)')
fig.colorbar(im, ax=axes[0])
im = axes[1].imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[1].set_xticks(range(corr.shape[1])); axes[1].set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=FS_ANNOT_SM)
axes[1].set_yticks(range(corr.shape[0])); axes[1].set_yticklabels(corr.index)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        axes[1].text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=6)
axes[1].set_title('Pearson r, W vs BIAS10 (HC)')
fig.colorbar(im, ax=axes[1])
fig.tight_layout()
fig.savefig(FIG_DIR / f'covariate_vif_N{N_MAIN}.png', dpi=300, bbox_inches='tight')
plt.show()

k,0,1,2,3,4,5,6,7,8,9,10
log(Total Reads),1.33,1.33,1.33,1.35,1.37,2.10,2.20,2.24,2.31,2.35,2.36
Spliced Reads (%),2.46,2.46,2.62,2.62,2.64,2.69,2.75,2.77,3.43,4.05,4.05
gDNA Contamination (Intron/Exon),3.50,3.79,3.84,3.85,3.86,4.07,4.08,4.08,4.18,4.18,4.22
rRNA Fraction,1.62,1.72,1.73,1.85,1.85,1.98,2.03,2.05,2.05,2.06,2.13
RNA Degradation (3' Bias),1.14,1.20,1.66,1.78,2.19,2.21,2.22,2.35,3.07,4.95,5.39
Platelet Score,2.71,4.07,8.79,17.47,17.64,17.77,18.47,29.63,33.79,34.43,38.65
GC Bias,1.84,2.08,2.12,4.10,4.11,4.11,4.14,5.29,5.48,5.56,5.56
Gene Length Bias,1.91,1.93,3.68,5.96,6.04,6.59,8.02,8.52,8.76,9.16,9.17
NG80,5.30,8.21,8.27,8.45,9.35,9.35,10.57,10.64,10.71,12.24,12.59
(NP80/NG80),2.40,2.41,2.50,3.11,3.32,3.54,3.57,3.57,3.58,3.60,3.63


/tmp/ipykernel_388147/1137095480.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
